###Anamoly Detection

###z-score anomaly detection
The z-score formula

(AmountUSD - mean_amt) / std_amt

This is:

z=x-u/o

Where:
x = transaction amount
μ = mean
σ = standard deviation

transactions whose amount is many standard deviations away from the average.

In [0]:
%sql
CREATE OR REPLACE TABLE finance_dev.gold_transaction_anomalies AS
WITH stats AS (
  SELECT
      AVG(AmountUSD) AS mean_amt,
      STDDEV(AmountUSD) AS std_amt
  FROM finance_dev.silver_transactions_enriched
)

SELECT
    t.*,
    ROUND((t.AmountUSD - s.mean_amt) / s.std_amt, 2) AS z_score
FROM finance_dev.silver_transactions_enriched t
CROSS JOIN stats s
WHERE ABS((t.AmountUSD - s.mean_amt) / s.std_amt) > 2.1;


select * from finance_dev.gold_transaction_anomalies

In [0]:
%sql
CREATE OR REPLACE TABLE finance_dev.gold_anomaly_explanations AS
SELECT
  TransactionID,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    concat(
      'You are a senior finance risk analyst. ',
        'Provide ONE concise business insight (max 25 words). ',
        'Avoid repeating the z-score definition. ',
        'Focus on what makes the transaction operationally or financially unusual. ',
        'Focus on concentration, currency exposure, customer exposure, and review priorities.',
        'Data: ',
      to_json(named_struct(
        'TransactionID', TransactionID,
        'Customer', CustomerName,
        'AmountUSD', AmountUSD,
        'Currency', Currency,
        'ZScore', z_score,
        'TransactionDate',TransactionDate,
        'BusinessUnit',BusinessUnit,
        'CostCenter',CostCenter,
        'CostCenterDepartment',CostCenterDepartment,
        'AccountType',AccountType,
        'ExchangeRate',ExchangeRate,
        'Amount',Amount,
        'Region',Region,
        'CustomerName',CustomerName
      ))
    )
  ) AS explanation
FROM finance_dev.gold_transaction_anomalies;

In [0]:
%sql
SELECT * FROM finance_dev.gold_anomaly_explanations

###Executive summary

In [0]:
%sql
CREATE OR REPLACE TABLE finance_dev.gold_anomaly_summary AS
SELECT
    COUNT(*) AS anomaly_count,
    ROUND(SUM(AmountUSD),2) AS total_anomalous_amount,
    ROUND(AVG(AmountUSD),2) AS avg_anomalous_amount,
    ROUND(MAX(AmountUSD),2) AS max_anomalous_amount,
    COUNT(DISTINCT CustomerName) AS affected_customers,
    COUNT(DISTINCT Currency) AS affected_currencies,
    COUNT(DISTINCT BusinessUnit) AS affected_business_units
FROM finance_dev.gold_transaction_anomalies;

select * from finance_dev.gold_anomaly_summary;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW anomaly_top_customers AS
SELECT
    CustomerName,
    COUNT(*) AS anomaly_txns,
    ROUND(SUM(AmountUSD),2) AS anomaly_amount
FROM finance_dev.gold_transaction_anomalies
GROUP BY CustomerName
ORDER BY anomaly_amount DESC;

select * from anomaly_top_customers



In [0]:
%sql
CREATE OR REPLACE TEMP VIEW anomaly_currency_mix AS
SELECT
    Currency,
    COUNT(*) AS txn_count,
    ROUND(SUM(AmountUSD),2) AS total_amount
FROM finance_dev.gold_transaction_anomalies
GROUP BY Currency
ORDER BY total_amount DESC;

select * from anomaly_currency_mix

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW anomaly_bu_mix AS
SELECT
    BusinessUnit,
    COUNT(*) AS txn_count,
    ROUND(SUM(AmountUSD),2) AS total_amount
FROM finance_dev.gold_transaction_anomalies
GROUP BY BusinessUnit
ORDER BY total_amount DESC;

select * from anomaly_bu_mix

In [0]:
%sql
CREATE OR REPLACE TABLE finance_dev.gold_anomaly_executive_summary AS
SELECT
  current_timestamp() AS GeneratedAt,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    concat(
      'You are a senior finance risk analyst preparing a management summary. ',
      'Generate exactly 5 concise bullet points (each under 18 words). ',
      'Do not repeat wording. ',
      'Focus on customer concentration, currency exposure, business-unit concentration, largest anomaly, and review priorities. ',
      'Overall summary: ',
      (SELECT to_json(named_struct(
          'anomaly_count', anomaly_count,
          'total_anomalous_amount', total_anomalous_amount,
          'avg_anomalous_amount', avg_anomalous_amount,
          'max_anomalous_amount', max_anomalous_amount,
          'affected_customers', affected_customers,
          'affected_currencies', affected_currencies,
          'affected_business_units', affected_business_units
      )) FROM anomaly_summary),
      '. Top customers: ',
      (SELECT to_json(collect_list(named_struct(
          'customer', CustomerName,
          'transactions', anomaly_txns,
          'amount', anomaly_amount
      ))) FROM anomaly_top_customers),
      '. Currency mix: ',
      (SELECT to_json(collect_list(named_struct(
          'currency', Currency,
          'transactions', txn_count,
          'amount', total_amount
      ))) FROM anomaly_currency_mix),
      '. Business-unit mix: ',
      (SELECT to_json(collect_list(named_struct(
          'business_unit', BusinessUnit,
          'transactions', txn_count,
          'amount', total_amount
      ))) FROM anomaly_bu_mix)
    )
  ) AS ExecutiveInsights;

select * from finance_dev.gold_anomaly_executive_summary;